# Mork-2 model and training summary

Builds the default `GPT` and reports parameter counts, a rough memory estimate, and the training schedule. Shard and tokenizer details live in `analyze_data.ipynb`.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mork.config import ModelConfig, TrainConfig
from mork.model import GPT
from mork.tokenizer import MorkTokenizer, SPECIAL_TOKENS

## Config and parameter count

In [2]:
model_cfg = ModelConfig()
train_cfg = TrainConfig()
model = GPT(model_cfg)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_embedding = total - model.wte.weight.numel()

print("ModelConfig")
for key, value in model_cfg.__dict__.items():
    print(f"  {key}: {value}")
print()
print(f"total parameters:      {total:,}")
print(f"trainable parameters:  {trainable:,}")
print(f"non-embedding params:  {non_embedding:,}  (excludes wte; RoPE is buffer-only)")
print(f"tied lm_head with wte: {model.wte.weight is model.lm_head.weight}")
print(f"head dim:              {model_cfg.n_embd // model_cfg.n_head}")
mlp_hidden = 64 * round((8 * model_cfg.n_embd) / (3 * 64))
print(f"MLP (SwiGLU) hidden:   {mlp_hidden}")
print(f"position encoding:     RoPE (theta={model_cfg.rope_theta:g})")

ModelConfig
  block_size: 512
  vocab_size: 16384
  n_layer: 12
  n_head: 12
  n_embd: 768
  dropout: 0.0
  rope_theta: 10000.0

total parameters:      97,536,768
trainable parameters:  97,536,768
non-embedding params:  84,953,856  (excludes wte; RoPE is buffer-only)
tied lm_head with wte: True
head dim:              64
MLP (SwiGLU) hidden:   2048
position encoding:     RoPE (theta=10000)


## Breakdown by module

In [3]:
rows = []
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    if name == "lm_head":
        n = 0
        note = "(tied with wte — not counted again)"
    else:
        note = ""
    rows.append((name, n, note))

block_params = [sum(p.numel() for p in blk.parameters()) for blk in model.h]

print(f"{'module':<12} {'params':>14}  note")
print("-" * 56)
for name, n, note in rows:
    print(f"{name:<12} {n:>14,}  {note}")
print("-" * 56)
print(f"{'TOTAL':<12} {total:>14,}")
print()
print(f"params / block: {block_params[0]:,}  × {len(block_params)} layers")
print(f"block total:    {sum(block_params):,}")

module               params  note
--------------------------------------------------------
wte              12,582,912  
drop                      0  
rotary                    0  
h                84,953,088  
ln_f                    768  
lm_head                   0  (tied with wte — not counted again)
--------------------------------------------------------
TOTAL            97,536,768

params / block: 7,079,424  × 12 layers
block total:    84,953,088


## Memory and compute estimates

In [4]:
def bytes_for(params: int, bytes_per: int) -> str:
    mb = params * bytes_per / (1024**2)
    return f"{mb:,.1f} MiB"

print("Weights only (approx)")
print(f"  fp32:      {bytes_for(total, 4)}")
print(f"  bf16/fp16: {bytes_for(total, 2)}")
print()
print("AdamW state (fp32 master plus two moment buffers) on top of bf16 weights:")
adam_extra = total * (4 + 4 + 4)
print(f"  optimizer extras ≈ {adam_extra / (1024**2):,.1f} MiB")
print(f"  rough train footprint (bf16 weights + Adam) ≈ {(total * 2 + adam_extra) / (1024**2):,.1f} MiB")
print("  activations and gradients are extra and dominate at long sequences")
print()
flops_per_token = 6 * non_embedding
tokens_per_step = train_cfg.total_batch_size
print(f"~FLOPs / token (non-emb ≈ 6N): {flops_per_token / 1e9:.2f} GFLOPs")
print(f"~FLOPs / optimizer step:       {flops_per_token * tokens_per_step / 1e12:.2f} TFLOPs")

Weights only (approx)
  fp32:      372.1 MiB
  bf16/fp16: 186.0 MiB

AdamW state (fp32 master plus two moment buffers) on top of bf16 weights:
  optimizer extras ≈ 1,116.2 MiB
  rough train footprint (bf16 weights + Adam) ≈ 1,302.3 MiB
  activations and gradients are extra and dominate at long sequences

~FLOPs / token (non-emb ≈ 6N): 0.51 GFLOPs
~FLOPs / optimizer step:       16.70 TFLOPs


## Training schedule

`max_steps` is the short trainer default. The corpus is about 2 billion tokens, so a real pretrain sets a longer step count on the CLI.

In [5]:
print("TrainConfig")
for key in [
    "batch_size",
    "seq_len",
    "total_batch_size",
    "grad_accum_steps",
    "max_lr",
    "min_lr",
    "min_lr_ratio",
    "warmup_steps",
    "max_steps",
    "weight_decay",
    "grad_clip",
    "dtype",
    "compile",
    "shuffle_chunk_tokens",
]:
    if key == "grad_accum_steps":
        value = train_cfg.grad_accum_steps
    elif key == "min_lr":
        value = train_cfg.min_lr
    else:
        value = getattr(train_cfg, key)
    print(f"  {key}: {value}")

print(f"  dropout: {model_cfg.dropout}")
tokens_seen = train_cfg.max_steps * train_cfg.total_batch_size
print()
print(f"tokens / micro-batch: {train_cfg.batch_size * train_cfg.seq_len:,}")
print(f"tokens / step:        {train_cfg.total_batch_size:,}")
print(f"tokens over default run: {tokens_seen:,}  ({tokens_seen / 1e6:.1f}M)")
print(f"passes over ~2.0B train tokens at that budget: {tokens_seen / 1.995e9:.2f}")

TrainConfig
  batch_size: 16
  seq_len: 512
  total_batch_size: 32768
  grad_accum_steps: 4
  max_lr: 0.0008
  min_lr: 0.00016
  min_lr_ratio: 0.2
  warmup_steps: 100
  max_steps: 4000
  weight_decay: 0.1
  grad_clip: 1.0
  dtype: bfloat16
  compile: True
  shuffle_chunk_tokens: 32768
  dropout: 0.0

tokens / micro-batch: 8,192
tokens / step:        32,768
tokens over default run: 131,072,000  (131.1M)
passes over ~2.0B train tokens at that budget: 0.07


## Tokenizer check

In [6]:
tok_path = ROOT / "data" / "tokenizer.json"
print(f"tokenizer: {tok_path}")
if tok_path.exists():
    tok = MorkTokenizer.load(tok_path)
    print(f"  vocab_size: {tok.vocab_size:,}")
    print(f"  specials:   {list(SPECIAL_TOKENS)}")
    for name in SPECIAL_TOKENS:
        print(f"    {name} -> {tok.special_token_id(name)}")
    print(f"  matches ModelConfig.vocab_size: {tok.vocab_size == model_cfg.vocab_size}")
else:
    print("  (missing — run mork-prepare)")

tokenizer: /home/mart/projects/llms/mork-2/data/tokenizer.json
  vocab_size: 16,384
  specials:   ['<|user|>', '<|assistant|>', '<|end|>', '<|pad|>']
    <|user|> -> 0
    <|assistant|> -> 1
    <|end|> -> 2
    <|pad|> -> 3
  matches ModelConfig.vocab_size: True


## Checkpoint, if one exists

In [7]:
from mork.checkpoint import best_val_checkpoint_path, latest_checkpoint_path

CKPT_DIR = ROOT / "checkpoints"
path = best_val_checkpoint_path(CKPT_DIR) or latest_checkpoint_path(CKPT_DIR)

if path is None:
    print(f"no checkpoint in {CKPT_DIR}")
else:
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    ckpt_cfg = ModelConfig(**ckpt["model_config"])
    print(f"loaded {path}")
    print(f"  step: {ckpt['step']}")
    print(f"  loss: {ckpt.get('val_loss', ckpt.get('loss'))}")
    print(f"  model_config: {ckpt_cfg}")
    print(f"  matches current ModelConfig: {ckpt_cfg == model_cfg}")
    n_ckpt = sum(v.numel() for v in ckpt["model"].values())
    print(f"  state_dict tensors: {len(ckpt['model'])}, numel sum: {n_ckpt:,}")

no checkpoint in /home/mart/projects/llms/mork-2/checkpoints
